# 07 · Benchmark: does a standard strong EEG decoder escape the category confound?

**Added after the H2 results; settings fixed in `config.yaml` (`benchmark`) before this notebook first ran.**

NB5 found that spectral features decode report embeddings above a free-shuffle null, but not beyond what site
and awakening type explain. Is that a weakness of the spectral features, or of the data? This notebook swaps
in a standard strong EEG representation and keeps everything else identical to NB5:

- **Decoder:** Riemannian tangent space (Barachant et al. 2012) + the same ridge. Per epoch, the 16 × 16
  channel covariance of the cleaned signal (non-rejected samples only, OAS shrinkage: the average reference and
  interpolation make raw covariances singular), projected to the tangent space at the Riemannian mean of the
  **training participants only**, refitted in every fold.
- **Same as NB5:** the 101 awakenings, leave-one-participant-out folds, SBERT targets, rank-accuracy score,
  1000 free-shuffle permutations (the same permutations), and 1000 permutations within site × awakening type.

Reading: above the free-shuffle null but at chance within category → the confound belongs to the data, not to
one feature set. Above the within-category null → content information that NB5's features missed.

In [ ]:
# Bootstrap: same cell in every notebook. On Colab: mount Drive, get the (public) repo, install extras.
import os, sys, subprocess
REPO_URL = "https://github.com/sanjanasinghx/dream2image-es606.git"
if "google.colab" in sys.modules and not os.environ.get("D2I_HEADLESS"):  # headless = run by d2i.runner
    from google.colab import drive
    drive.mount("/content/drive")
    REPO = "/content/dream2image-es606"
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
    else:
        subprocess.run(["git", "-C", REPO, "fetch", "-q", "origin"], check=True)
        subprocess.run(["git", "-C", REPO, "reset", "-q", "--hard", "origin/main"], check=True)
    freeze = lambda: subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
    before = freeze()
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{REPO}/requirements-colab.txt"], check=True)
    if freeze() != before:  # e.g. numpy upgraded under an already-loaded copy -> broken imports
        print("Packages changed: restarting the runtime. When it reconnects, Run all again.")
        os.kill(os.getpid(), 9)
else:
    REPO = os.path.abspath("..")
sys.path.insert(0, REPO)
from d2i import env
cfg = env.load_config()
rng = env.seed_everything(cfg["seed"])
print("data root:", env.data_root())

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from d2i import decode, stats
dc, bc = cfg["h2"], cfg["benchmark"]
der, tables, figs = env.path(cfg, "derived"), env.path(cfg, "results") / "tables", env.path(cfg, "results") / "figures"
tables.mkdir(exist_ok=True); figs.mkdir(exist_ok=True)
X, meta = decode.feature_matrices(pd.read_csv(der / "features_long.csv"), dc["epochs"], dc["features"])  # NB5's rows
z = np.load(der / "report_embeddings.npz", allow_pickle=True)
E = dict(zip(z["sample_id"], z["emb"]))
Y = np.array([E[s] for s in meta.sample_id])
groups = meta.participant.values
print("awakenings", len(Y), "| participants", meta.participant.nunique())

## 1. Channel covariances of the cleaned epochs

In [ ]:
covs, chans = {e: [] for e in dc["epochs"]}, None
for s in meta.sample_id:
    c = np.load(der / "clean" / f"{s}.npz")
    chans = chans or list(c["channels"])
    assert list(c["channels"]) == chans, s
    for e in dc["epochs"]:
        covs[e].append(decode.clean_covariance(c[e], c[f"{e}_bad"], bc["cov_estimator"]))
covs = {e: np.array(v) for e, v in covs.items()}
for e, C in covs.items():
    print(e, C.shape, "smallest eigenvalue", f"{np.linalg.eigvalsh(C).min():.2e}")

## 2. Decoding against both nulls

In [ ]:
rng = np.random.default_rng(cfg["seed"])
perms = [rng.permutation(len(Y)) for _ in range(dc["n_perm"])]  # same draws as NB5's free shuffle
meta["stratum"] = meta.site + "_" + meta.tag_class.fillna("case")
strata = list(meta.groupby("stratum").indices.values())
srng = np.random.default_rng(cfg["seed"] + 1)
def strat_perm():
    p = np.arange(len(Y))
    for idx in strata:
        p[idx] = srng.permutation(idx)
    return p
sperms = [strat_perm() for _ in range(dc["n_perm"])]
rows = []
for e in dc["epochs"]:
    folds = decode.make_folds(covs[e], groups, fit_transform=decode.tangent_space(covs[e]))
    obs = decode.loso_rank(folds, Y, dc["alphas"])
    for null_name, P in [("free shuffle", perms), ("within site x awakening type", sperms)]:
        null = np.array([decode.loso_rank(folds, Y[p], dc["alphas"]) for p in P])
        rows.append(decode.vs_null(obs, null, {"decoder": "Riemannian tangent space", "epoch": e, "null": null_name}))
        print({k: (round(v, 4) if isinstance(v, float) else v) for k, v in rows[-1].items()})

## 3. Side by side with NB5 (spectral features) and the category-only baseline

In [ ]:
nb5 = pd.read_csv(tables / "h2_decoding_by_epoch.csv").assign(decoder="spectral features (NB5)", null="free shuffle")
conf = pd.read_csv(tables / "h2_confound_control.csv")
within = conf[conf.model.str.startswith("EEG")].assign(decoder="spectral features (NB5)",
                                                         epoch=lambda d: d.model.str[-2:])
cat = conf[conf.model.str.startswith("category")].assign(decoder="category labels only (no EEG)", epoch="-")
cols = ["decoder", "epoch", "null", "rank_acc", "null_mean", "above_chance", "null_p95", "p_perm", "n"]
bench = pd.concat([pd.DataFrame(rows), nb5, within, cat])[cols].reset_index(drop=True)
for (d, n), g in bench.groupby(["decoder", "null"]):
    bench.loc[g.index, "p_holm"] = stats.holm(g.p_perm)  # across the 3 epochs of each decoder x null
bench.to_csv(tables / "benchmark_h2.csv", index=False)
print(bench.round(4).to_string(index=False))

## 4. Figure

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
dec = [("spectral features (NB5)", "#4a6fa5"), ("Riemannian tangent space", "#2ca25f")]
x = np.arange(len(dc["epochs"]))
for a, null_name in zip(ax, ["free shuffle", "within site x awakening type"]):
    for k, (d, c) in enumerate(dec):
        g = bench[(bench.decoder == d) & (bench.null == null_name)].set_index("epoch").loc[dc["epochs"]]
        a.bar(x + (k - .5) * .38, g.above_chance, width=.36, color=c, label=d)
        for xi, (v, p) in enumerate(zip(g.above_chance, g.p_holm)):
            a.text(xi + (k - .5) * .38, max(v, 0) + .004, f"p={p:.3f}", ha="center", fontsize=7)
    a.axhline(0, color="k", lw=.8)
    a.set(xticks=x, xticklabels=dc["epochs"], title=f"null: {null_name}")
ax[0].axhline(cat.above_chance.iloc[0], color="#e08214", ls="--", lw=1, label="category labels alone, no EEG")
ax[0].set(ylabel="rank accuracy above the null mean"); ax[0].legend(fontsize=8)
fig.suptitle("Benchmark (added after H2): EEG -> report embedding, leave-one-participant-out, Holm p over epochs")
fig.tight_layout(); fig.savefig(figs / "benchmark_h2.pdf"); fig.savefig(figs / "benchmark_h2.png", dpi=150)
env.write_manifest(cfg, "07_benchmark", [tables / "benchmark_h2.csv"])